In [3]:
import pandas as pd
import numpy as np

# -----------------------------
# Load factor file
# -----------------------------
df = pd.read_csv("factor_data.csv")

df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df = df.sort_values("Date").reset_index(drop=True)

# Start analysis here
start_date = pd.Timestamp("2010-01-31")
df = df[df["Date"] >= start_date].copy()

factor_cols = ["MKT", "SMB", "HML", "RMW", "CMA", "MOM", "RF", "RM"]

print("Shape:", df.shape)
print("Date range:", df["Date"].min(), "to", df["Date"].max())
print("\nMissing values:")
print(df[factor_cols].isna().sum())

# -----------------------------
# Summary statistics
# -----------------------------
summary = df[factor_cols].agg(["mean", "std", "min", "max"]).T
summary["mean_annualized"] = summary["mean"] * 12
summary["std_annualized"] = summary["std"] * np.sqrt(12)

print("\n=== SUMMARY STATS ===")
print(summary.round(4).to_string())

# -----------------------------
# Check that MKT = RM - RF
# -----------------------------
df["MKT_check"] = df["RM"] - df["RF"]
df["MKT_diff"] = df["MKT"] - df["MKT_check"]

print("\n=== MKT CHECK ===")
print("Max abs difference:", df["MKT_diff"].abs().max())

# -----------------------------
# Correlation matrix
# -----------------------------
print("\n=== CORRELATION MATRIX ===")
print(df[factor_cols].corr().round(3).to_string())

# -----------------------------
# Biggest absolute months for each factor
# -----------------------------
print("\n=== LARGEST ABSOLUTE MONTHS ===")
for col in ["MKT", "SMB", "HML", "RMW", "CMA", "MOM"]:
    tmp = df[["Date", col]].dropna().copy()
    tmp["abs_val"] = tmp[col].abs()
    top = tmp.sort_values("abs_val", ascending=False).head(5).drop(columns="abs_val")
    print(f"\n{col}:")
    print(top.to_string(index=False))

# -----------------------------
# Cumulative growth of 1 NOK
# -----------------------------
cum = (1 + df[["Date", "MKT", "SMB", "HML", "RMW", "CMA", "MOM"]].set_index("Date")).cumprod()

print("\n=== FINAL CUMULATIVE VALUES ===")
print(cum.tail(1).round(3).to_string())

# -----------------------------
# Quick view of first/last rows
# -----------------------------
print("\n=== FIRST 10 ROWS ===")
print(df.head(10).to_string(index=False))

print("\n=== LAST 10 ROWS ===")
print(df.tail(10).to_string(index=False))

Shape: (195, 9)
Date range: 2010-01-31 00:00:00 to 2026-03-31 00:00:00

Missing values:
MKT    0
SMB    0
HML    0
RMW    0
CMA    0
MOM    0
RF     0
RM     0
dtype: int64

=== SUMMARY STATS ===
       mean     std     min     max  mean_annualized  std_annualized
MKT  0.0055  0.0356 -0.0986  0.0969           0.0658          0.1234
SMB  0.0002  0.0268 -0.0701  0.0862           0.0025          0.0928
HML -0.0014  0.0306 -0.1110  0.0923          -0.0172          0.1061
RMW  0.0024  0.0276 -0.0630  0.0716           0.0289          0.0957
CMA  0.0002  0.0267 -0.0598  0.0716           0.0021          0.0926
MOM  0.0136  0.0323 -0.0923  0.1055           0.1632          0.1119
RF   0.0016  0.0011  0.0001  0.0039           0.0196          0.0039
RM   0.0071  0.0355 -0.0949  0.0971           0.0853          0.1230

=== MKT CHECK ===
Max abs difference: 1.1102230246251565e-16

=== CORRELATION MATRIX ===
       MKT    SMB    HML    RMW    CMA    MOM     RF     RM
MKT  1.000  0.067  0.069 -0.074 -